# Figures 8, 9 and 10 -- composite eddy structure

Every eddy is normalised by its own radius before averaging, giving the mean
horizontal structure on lead day 1 and the change by lead days 5, 10 and 15.
Figures 8 and 9 map that composite for sea surface height and for geostrophic
speed; Figure 10 reduces it to radial profiles.

Three things the files record that the panels do not show on their own:

* `*_reference` is the lead-day-1 composite of **the eddies surviving to that day**,
  which is the baseline each change is differenced against. Comparing it with
  `*_day1` separates a real change from the shrinking population.
* the percentage curves of Figure 10 are stored already masked, blank exactly where
  the figure is blank, because the ratio diverges as the lead-day-1 profile
  approaches zero -- at large radius for sea surface height, at small radius for
  speed.
* `n_eddies` gives the count behind each column.

**What this reproduces.** The published figures place the forecast beside an
equally selected set of eddies detected in the mapped altimetry, which acts as a
survivorship control. If that product was archived it is drawn too; otherwise only
the forecast rows appear and the control is not available. The notebook reports
which case it is in.


In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# Folder holding the archived NetCDF files. Default: alongside this notebook.
DATA = Path(".")

FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)

REGION_ORDER = ["Kuroshio", "Gulf of Mexico", "Arabian Sea"]


def by_region(paths):
    """Open each file, read its region attribute, return them in paper order."""
    tagged = []
    for p in paths:
        with xr.open_dataset(p) as ds:
            tagged.append((ds.attrs.get("region", p.stem), p))
    key = lambda t: (REGION_ORDER.index(t[0]) if t[0] in REGION_ORDER else 99, t[0])
    return sorted(tagged, key=key)


def need(pattern):
    found = sorted(DATA.glob(pattern))
    if not found:
        raise FileNotFoundError(
            f"No file matching {pattern!r} in {DATA.resolve()}. "
            f"Set DATA to the folder holding the archived NetCDF files.")
    return found


In [ ]:
CLIM = {("ssh",   "day1"): {"Kuroshio": (-20, 20), "Gulf of Mexico": (-10, 10),
                            "Arabian Sea": (-10, 10)},
        ("ssh",   "diff"): {"Kuroshio": (-2, 2), "Gulf of Mexico": (-2, 2),
                            "Arabian Sea": (-2, 2)},
        ("speed", "day1"): {"Kuroshio": (0, 30), "Gulf of Mexico": (0, 30),
                            "Arabian Sea": (0, 30)},
        ("speed", "diff"): {"Kuroshio": (-3, 3), "Gulf of Mexico": (-3, 3),
                            "Arabian Sea": (-3, 3)}}
CMAP_DAY1 = {"ssh": "RdBu_r", "speed": "jet"}
CMAP_DIFF = "seismic"
VARLAB = {"ssh": "SSH (cm)", "speed": "Speed (cm s$^{-1}$)"}


def limits(var, group, region, ds):
    tab = CLIM.get((var, group), {})
    if region in tab:
        return tab[region]
    key = f"{var}_day1" if group == "day1" else f"{var}_change"
    m = float(np.nanpercentile(np.abs(ds[key].values), 99.5))
    return (0.0, m) if (group == "day1" and var == "speed") else (-m, m)


def composite_figure(ds, region, var):
    prods = list(ds["product"].values)
    rows = [(po, pr) for po in ds.polarity.values for pr in prods]
    cols = [None] + list(ds.diff_day.values)
    fig, axes = plt.subplots(len(rows), len(cols),
                             figsize=(1.65 * len(cols) + 0.9, 1.75 * len(rows) + 0.9),
                             dpi=130, squeeze=False)
    pcm = {}
    for r, (po, pr) in enumerate(rows):
        for c, d in enumerate(cols):
            ax = axes[r][c]
            if d is None:
                z = ds[f"{var}_day1"].sel(product=pr, polarity=po)
                lo, hi = limits(var, "day1", region, ds)
                cm, grp = CMAP_DAY1[var], "day1"
            else:
                z = ds[f"{var}_change"].sel(product=pr, polarity=po, diff_day=d)
                lo, hi = limits(var, "diff", region, ds)
                cm, grp = CMAP_DIFF, "diff"
            pcm[grp] = ax.pcolormesh(ds.x, ds.y, z, cmap=cm, vmin=lo, vmax=hi,
                                     shading="auto")
            ax.add_patch(plt.Circle((0, 0), 1.0, fill=False, color="0.3", lw=0.6))
            ax.add_patch(plt.Circle((0, 0), 1.5, fill=False, color="0.3", lw=1.0))
            ax.set_aspect("equal")
            ax.set_xticks([]); ax.set_yticks([])
            ax.text(0.03, 0.96, f"({chr(97 + r * len(cols) + c)})",
                    transform=ax.transAxes, fontsize=7, fontweight="bold",
                    va="top")
            if r == 0:
                ax.set_title("Day 1" if d is None else f"Day {int(d)} $-$ Day 1",
                             fontsize=8)
            if c == 0:
                lab = str(po).capitalize()
                if len(prods) > 1:
                    lab += f"\n{str(pr)}"
                ax.set_ylabel(lab, fontsize=7.5)
    for grp, ax_list, lab in (("day1", [a[0] for a in axes], VARLAB[var]),
                              ("diff", [a[-1] for a in axes],
                               f"Change in {VARLAB[var]}")):
        cb = fig.colorbar(pcm[grp], ax=ax_list, orientation="vertical",
                          fraction=0.03, pad=0.02)
        cb.set_label(lab, fontsize=7.5)
        cb.ax.tick_params(labelsize=6.5)
    fig.suptitle(f"{region} -- composite {VARLAB[var].split(' (')[0].lower()}",
                 fontsize=10, fontweight="bold")
    return fig


for region, path in by_region(need("Figures8_9_composite_*.nc")):
    with xr.open_dataset(path) as ds:
        if len(ds["product"].values) == 1:
            print(f"{region}: forecast only -- the survivorship control is not "
                  f"in this deposit")
        for figno, var in ((8, "ssh"), (9, "speed")):
            fig = composite_figure(ds, region, var)
            out = FIGDIR / f"Figure{figno}_composite_{var}_{region.replace(' ', '_')}.png"
            fig.savefig(out, dpi=300, bbox_inches="tight")
            plt.show()
            print("wrote", out)


In [ ]:
# ---- Figure 10: radial profiles ------------------------------------------
PROF_COLOR = {"anticyclonic": "#C0392B", "cyclonic": "#1B6CA8"}
PROF_STYLE = {"observed": dict(ls="-.", lw=1.1), "forecast": dict(ls="-", lw=1.4)}

path = need("Figure10_radial_profiles.nc")[0]
with xr.open_dataset(path) as ds:
    DAY_N = 15
    ROWS = [("ssh_day1_profile",     "Day 1 SSH (cm)"),
            ("ssh_change_percent",   f"SSH change by day {DAY_N}\n(% of day 1)"),
            ("speed_day1_profile",   "Day 1 speed (cm s$^{-1}$)"),
            ("speed_change_percent", f"Speed change by day {DAY_N}\n(% of day 1)")]
    regions = list(ds.region.values)
    fig, axes = plt.subplots(len(ROWS), len(regions), figsize=(7.0, 8.2),
                             dpi=130, squeeze=False)
    li = 0
    for i, (v, ylab) in enumerate(ROWS):
        for j, rg in enumerate(regions):
            ax = axes[i][j]
            for pr in ds["product"].values:
                for po in ds.polarity.values:
                    ax.plot(ds.r, ds[v].sel(region=rg, product=pr, polarity=po),
                            color=PROF_COLOR[str(po)],
                            label=f"{str(po)[:2].upper()}, {str(pr)}",
                            **PROF_STYLE[str(pr)])
            ax.axhline(0, color="0.6", lw=0.5, zorder=0)
            ax.axvline(1.0, color="0.6", lw=0.5, ls=":", zorder=0)
            ax.grid(ls="--", lw=0.3)
            ax.tick_params(labelsize=6.5)
            if v.endswith("percent"):
                ax.set_xlim(0, 1.0)
            ax.text(0.03, 0.95, f"({chr(97 + li)})", transform=ax.transAxes,
                    fontsize=7.5, fontweight="bold", va="top")
            li += 1
            if i == 0:
                ax.set_title(str(rg), fontsize=9, fontweight="bold")
            if j == 0:
                ax.set_ylabel(ylab, fontsize=7.5)
            if i == len(ROWS) - 1:
                ax.set_xlabel("$r/R$", fontsize=8)
    h, l = axes[0][0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=len(l), fontsize=7,
               frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.tight_layout(h_pad=0.5, w_pad=0.4, rect=(0, 0.02, 1, 1))
    out = FIGDIR / "Figure10_radial_profiles.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("wrote", out)
    print("\npercentage mask:", ds.attrs.get("percentage_mask", "(not recorded)"))
